# Project 17 — Nonparametric Curves with a Gaussian Process

**Scenario.** A thermal-melt / dose-response experiment yields a smooth response $y$ at inputs $x$ (temperature or log-dose). We do **not** want to commit to a parametric shape (logistic, polynomial). A **Gaussian process** (GP) lets the data choose the curve while quantifying uncertainty.

**New skill.** Kernels, hyperpriors, and *flexibility control*. **Key pitfall.** The length-scale $\ell$ and the marginal variance $\eta^2$ trade off against each other (and against the noise $\sigma$). A vague length-scale prior makes the model non-identifiable and the sampler unhappy. The cure is an **informative length-scale prior**.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

Each observation is $y_i = f(x_i) + \varepsilon_i$ with $\varepsilon_i \sim \mathcal{N}(0,\sigma^2)$ and $f$ a smooth latent function. **Assumptions made explicit:** (a) $f$ is smooth (the ExpQuad kernel encodes infinite differentiability), (b) noise is homoscedastic Gaussian, (c) inputs are noise-free. We synthesize from a known curve and a known $\sigma=0.18$ so we can check recovery of both the curve and the noise.

In [ ]:
from data.generate_data import generate, f_true
data = generate()
x, y = data['x'], data['y']
print(f"n={data['n']}, true sigma={data['truth']['sigma']}")
fig, ax = plt.subplots(figsize=(6,3.5))
ax.plot(x, data['f_true'], 'k--', label='true f(x)')
ax.scatter(x, y, s=18, color='#4C72B0', label='noisy data')
ax.set(xlabel='x (e.g. temperature)', ylabel='response', title='Data and latent curve')
ax.legend(); plt.tight_layout()

## Step 2 — Model specification (kernel + justified hyperpriors)

$$f \sim \mathcal{GP}(0,\; k),\quad k(x,x') = \eta^2\exp\!\Big(-\tfrac{(x-x')^2}{2\ell^2}\Big),\quad y \sim \mathcal{N}(f(x),\sigma).$$

**Hyperpriors.** $\ell \sim \text{InverseGamma}(6,12)$ is the load-bearing choice: its mass sits near $\ell\approx 2$ (about a fifth of the input range), with little mass below 1 or above 5. This **informative** prior prevents the $\ell$/$\eta$ trade-off. $\eta \sim \text{HalfNormal}(2)$ controls amplitude; $\sigma \sim \text{HalfNormal}(0.5)$ is the noise. We use `pm.gp.Marginal`, which integrates $f$ out analytically — cheap and stable.

In [ ]:
from model import build_model, fit, predict_curve
model, gp = build_model(data)
model

## Step 3 — Prior predictive checks

We draw curves implied by the prior. We want functions that are wiggly on the scale of the data — not flat lines (length-scale too large) nor white noise (length-scale too small). The InverseGamma length-scale prior should yield smooth-but-non-trivial curves spanning a plausible response range.

In [ ]:
rng = np.random.default_rng(RNG)
xx = np.linspace(0, 10, 80)
fig, ax = plt.subplots(figsize=(6,3.5))
for _ in range(8):
    ell = 1.0/rng.gamma(6.0, 1.0/12.0)
    eta = abs(rng.normal(0, 2.0))
    d2 = (xx[:,None]-xx[None,:])**2
    K = eta**2*np.exp(-0.5*d2/ell**2) + 1e-8*np.eye(len(xx))
    f = rng.multivariate_normal(np.zeros(len(xx)), K)
    ax.plot(xx, f, lw=1)
ax.set(xlabel='x', ylabel='f(x)', title='Prior predictive draws — smooth, plausible')
plt.tight_layout()

## Step 4 — Inference (NUTS)

We sample the **hyperposterior** over $(\ell,\eta,\sigma)$ with NUTS. Settings: `draws=500, tune=1000, chains=2, target_accept=0.95`. GP hyperposteriors have mildly curved geometry, so a higher `target_accept` keeps divergences away. (Compute note: GP sampling is the heaviest in the portfolio; we keep $N$ small.)

In [ ]:
idata = fit(data, draws=500, tune=1000, chains=2, seed=101)

## Step 5 — Computational diagnostics

Check $\hat R \approx 1.00$, healthy ESS, and **zero divergences**. With the informative length-scale prior the pair plot of $(\ell,\eta)$ should be a compact blob, not a diagonal ridge. A ridge would signal the trade-off pathology (see the broken notebook).

In [ ]:
print(az.summary(idata, var_names=['ell','eta','sigma']))
n_div = int(idata.sample_stats['diverging'].sum())
print(f'divergences: {n_div}')

In [ ]:
az.plot_trace(idata, var_names=['ell','eta','sigma']); plt.tight_layout()

In [ ]:
az.plot_pair(idata, var_names=['ell','eta'], kind='scatter',
             scatter_kwargs={'alpha':0.3}); plt.tight_layout()

## Step 6 — Posterior predictive checks (the fitted curve)

We reconstruct the latent function on a dense grid using `gp.predict` averaged over posterior hyperparameter draws, and plot the posterior mean with a 94% credible band. A good fit: the band hugs the data, contains the **true** curve, and widens where data are sparse.

In [ ]:
pred = predict_curve(data, idata, n_pred=80, seed=3)
fig, ax = plt.subplots(figsize=(6.5,3.8))
ax.fill_between(pred['x_new'], pred['lower'], pred['upper'], color='#4C72B0',
                alpha=0.25, label='94% band')
ax.plot(pred['x_new'], pred['mean'], color='#4C72B0', label='GP mean')
ax.plot(data['x'], data['f_true'], 'k--', label='true f(x)')
ax.scatter(data['x'], data['y'], s=14, color='black', alpha=0.5, label='data')
ax.set(xlabel='x', ylabel='response', title='GP fit vs truth')
ax.legend(); plt.tight_layout()

In [ ]:
mae = float(np.mean(np.abs(predict_curve(data, idata, x_new=data['x'], seed=3)['mean']
                          - data['f_true'])))
print(f'curve recovery MAE at training inputs = {mae:.3f} (want < 0.25)')

## Step 7 — Model criticism & comparison

We criticise the fit by (a) confirming $\sigma$ recovers the truth and (b) checking residuals look like white noise of the inferred scale. A GP is a single flexible model; comparison against, say, a parametric logistic could be done with LOO, but the GP's value is precisely that it avoids that commitment.

In [ ]:
resid = data['y'] - predict_curve(data, idata, x_new=data['x'], seed=3)['mean']
print(f"residual sd = {resid.std():.3f}  vs  true sigma = {data['truth']['sigma']}")
post_sigma = idata.posterior['sigma'].values.ravel()
print(f"posterior sigma mean = {post_sigma.mean():.3f}, "
      f"94% = [{np.percentile(post_sigma,3):.3f}, {np.percentile(post_sigma,97):.3f}]")

## Step 8 — Decision & communication

Turn the curve into something actionable: e.g. the input $x$ at which the response crosses a threshold (a melt midpoint / an EC50-like quantity), with uncertainty. Here we report the posterior over the input where the mean curve first exceeds 1.0.

In [ ]:
xx = np.linspace(0, 10, 200)
p = predict_curve(data, idata, x_new=xx, seed=9)
cross = xx[np.argmax(p['mean'] > 1.0)]
print(f'Estimated input where response crosses 1.0: x = {cross:.2f}')
print('Communicate: report this crossing point WITH the credible band width '
      'there, not as a bare number.')

**Conclusion (for a collaborator).** The response rises smoothly with a clear transition; the GP recovers the latent curve to within MAE < 0.25 and the noise scale within tolerance. The length-scale prior — not the kernel choice — is what made this stable; see `PRIOR_SENSITIVITY.md`.